# Production Inference Test Pipeline (Real Data)

End-to-end notebook for production-like flow:
1. Load 20 sample images from `ml/test-data`.
2. Embed image using fine-tuned CLIP checkpoint (`ml/checkpoints/best_clip_model.pt`).
3. Generate concise metadata via Ollama Cloud API VLM.
4. Embed metadata using the same fine-tuned CLIP model.
5. Store in FAISS index + JSON payload (metadata JSONB-style structure).
6. Search flow: image rank first, then metadata re-rank.
7. RAG response using Ollama LLM over retrieved metadata.


In [ ]:
# If needed, install dependencies once:
# %pip install -q openai-clip faiss-cpu pillow ollama numpy pandas torch


In [13]:
from __future__ import annotations

import base64
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import clip
import faiss
import numpy as np
import pandas as pd
from ollama import Client
import torch
from PIL import Image

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = Path.cwd().resolve()
TEST_DATA_DIR = ROOT / "test-data"
CHECKPOINT_PATH = ROOT / "checkpoints" / "best_clip_model.pt"

# Ollama Cloud config
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "https://ollama.com")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "")

# Offline/high-quality VLM for metadata extraction
VLM_MODEL = os.getenv("OLLAMA_VLM_MODEL", "llava")

# Fast production LLM for RAG answer generation
RAG_LLM_MODEL = os.getenv("OLLAMA_RAG_LLM_MODEL", "gemma4:31b-cloud")

TOPK_STAGE1 = 10
TOPK_FINAL = 5

assert TEST_DATA_DIR.exists(), f"Missing test data folder: {TEST_DATA_DIR}"
assert CHECKPOINT_PATH.exists(), f"Missing checkpoint: {CHECKPOINT_PATH}"
print("Device:", DEVICE)
print("Test data dir:", TEST_DATA_DIR)
print("Checkpoint:", CHECKPOINT_PATH)
print("Ollama base URL:", OLLAMA_BASE_URL)


Device: cuda
Test data dir: D:\dev\visquery\ml\test-data
Checkpoint: D:\dev\visquery\ml\checkpoints\best_clip_model.pt
Ollama base URL: https://ollama.com


In [14]:
# Load fine-tuned CLIP model
model, preprocess = clip.load("ViT-B/32", device=DEVICE, jit=False)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = checkpoint.get("model_state_dict", checkpoint)
clean_state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)
model.eval()
print(f"Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")


C:\Users\shiva\AppData\Local\Temp\ipykernel_19532\1664199488.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE

Missing keys: 0 | Unexpected keys: 0


In [15]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def l2_normalize(vec: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(vec) + 1e-12
    return vec / norm

def embed_image(image_path: Path) -> np.ndarray:
    img = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = model.encode_image(img)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat[0].detach().cpu().numpy().astype(np.float32)

def embed_text(text: str) -> np.ndarray:
    tok = clip.tokenize([text], truncate=True).to(DEVICE)
    with torch.no_grad():
        feat = model.encode_text(tok)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat[0].detach().cpu().numpy().astype(np.float32)

def image_to_base64(image_path: Path) -> str:
    return base64.b64encode(image_path.read_bytes()).decode("utf-8")

def ollama_chat(model_name: str, messages: list[dict[str, Any]], temperature: float = 0.1) -> str:
    if not OLLAMA_API_KEY:
        raise RuntimeError("OLLAMA_API_KEY is empty. Set it in environment.")

    client = Client(
        host=OLLAMA_BASE_URL.strip().rstrip("/"),
        headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},
    )
    resp = client.chat(
        model=model_name,
        messages=messages,
        options={"temperature": temperature},
    )
    return resp.get("message", {}).get("content", "").strip()

def generate_metadata_with_vlm(image_path: Path) -> dict[str, Any]:
    prompt = (
        "Analyze this architecture image and return concise metadata as STRICT JSON with keys: "
        "style, era, materials, structure, ornaments, color_palette, location_hint, summary. "
        "Keep each field short and factual."
    )
    b64 = image_to_base64(image_path)
    messages = [
        {
            "role": "user",
            "content": prompt,
            "images": [b64],
        }
    ]
    raw = ollama_chat(VLM_MODEL, messages, temperature=0.0)

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {
            "style": "unknown",
            "era": "unknown",
            "materials": "unknown",
            "structure": "unknown",
            "ornaments": "unknown",
            "color_palette": "unknown",
            "location_hint": "unknown",
            "summary": raw[:600],
        }

def metadata_to_text(metadata: dict[str, Any]) -> str:
    ordered_keys = [
        "style", "era", "materials", "structure",
        "ornaments", "color_palette", "location_hint", "summary"
    ]
    parts = [f"{k}: {metadata.get(k, '')}" for k in ordered_keys]
    return " | ".join(parts)


In [16]:
# Load exactly 20 sample images
image_paths = sorted([p for p in TEST_DATA_DIR.glob("*") if p.suffix.lower() in IMG_EXTS])
assert len(image_paths) >= 20, f"Need at least 20 images in {TEST_DATA_DIR}, found {len(image_paths)}"
sample_paths = image_paths[:20]
print(f"Loaded sample images: {len(sample_paths)}")
sample_paths[:5]


Loaded sample images: 20


[WindowsPath('D:/dev/visquery/ml/test-data/AncientEgyptianArchitecture.webp'),
 WindowsPath('D:/dev/visquery/ml/test-data/AncientGreekArchitecture.webp'),
 WindowsPath('D:/dev/visquery/ml/test-data/Anglo-SaxonArchitecture-ezgif.com-optiwebp.webp'),
 WindowsPath('D:/dev/visquery/ml/test-data/Baroquearchitecture-ezgif.com-optiwebp-1-1.webp'),
 WindowsPath('D:/dev/visquery/ml/test-data/ByzantineArchitecture-ezgif.com-optiwebp.webp')]

In [17]:
@dataclass
class InferenceRecord:
    id: int
    image_path: str
    metadata_jsonb: dict[str, Any]
    image_embedding: np.ndarray
    metadata_embedding: np.ndarray

records: list[InferenceRecord] = []

for idx, p in enumerate(sample_paths):
    img_vec = embed_image(p)
    metadata = generate_metadata_with_vlm(p)
    meta_text = metadata_to_text(metadata)
    meta_vec = embed_text(meta_text)

    records.append(
        InferenceRecord(
            id=idx,
            image_path=str(p),
            metadata_jsonb=metadata,
            image_embedding=img_vec,
            metadata_embedding=meta_vec,
        )
    )

print(f"Built inference records: {len(records)}")


Built inference records: 20


In [18]:
# FAISS vector DB (in-memory)
# schema-like payload:
# inference_test
#   |- metadata JSONB
#   |- image_embedding vector
#   |- metadata_embedding vector

dim = records[0].image_embedding.shape[0]
index_image = faiss.IndexFlatIP(dim)
index_meta = faiss.IndexFlatIP(dim)

image_matrix = np.stack([r.image_embedding for r in records]).astype(np.float32)
meta_matrix = np.stack([r.metadata_embedding for r in records]).astype(np.float32)
index_image.add(image_matrix)
index_meta.add(meta_matrix)

payload_by_id = {r.id: {
    "id": r.id,
    "image_path": r.image_path,
    "metadata": r.metadata_jsonb,
} for r in records}

print("Image index ntotal:", index_image.ntotal)
print("Metadata index ntotal:", index_meta.ntotal)


Image index ntotal: 20
Metadata index ntotal: 20


In [19]:
def search_two_stage(query_image_path: Path, topk_stage1: int = TOPK_STAGE1, topk_final: int = TOPK_FINAL):
    # Stage 1: CLIP image retrieval
    q_img_vec = embed_image(query_image_path).reshape(1, -1).astype(np.float32)
    s1_scores, s1_ids = index_image.search(q_img_vec, topk_stage1)

    # Stage 2: metadata rerank using query metadata embedding
    q_meta = generate_metadata_with_vlm(query_image_path)
    q_meta_text = metadata_to_text(q_meta)
    q_meta_vec = embed_text(q_meta_text)

    candidates = []
    for rank, rid in enumerate(s1_ids[0]):
        if rid < 0:
            continue
        img_score = float(s1_scores[0][rank])
        meta_score = float(np.dot(q_meta_vec, records[rid].metadata_embedding))
        hybrid = 0.6 * img_score + 0.4 * meta_score
        candidates.append((rid, img_score, meta_score, hybrid))

    candidates.sort(key=lambda x: x[3], reverse=True)
    final_hits = candidates[:topk_final]

    rows = []
    for rid, img_score, meta_score, hybrid in final_hits:
        row = payload_by_id[rid].copy()
        row.update({
            "image_score": img_score,
            "metadata_score": meta_score,
            "hybrid_score": hybrid,
        })
        rows.append(row)

    return q_meta, pd.DataFrame(rows)


In [20]:
# Example inference query (pick one sample image or replace with new production image path)
query_path = sample_paths[0]
query_metadata, result_df = search_two_stage(query_path)
print("Query image:", query_path.name)
print("Query metadata:")
print(json.dumps(query_metadata, indent=2, ensure_ascii=False))
result_df[["id", "image_path", "hybrid_score", "image_score", "metadata_score"]]


Query image: AncientEgyptianArchitecture.webp
Query metadata:
{
  "style": "unknown",
  "era": "unknown",
  "materials": "unknown",
  "structure": "unknown",
  "ornaments": "unknown",
  "color_palette": "unknown",
  "location_hint": "unknown",
  "summary": "```json\n{\n  \"style\": \"Ancient Egyptian\",\n  \"era\": \"Antiquity\",\n  \"materials\": [\"Sandstone\", \"Limestone\"],\n  \"structure\": \"Monumental complex with pyramids, hypostyle halls, and colossal statues\",\n  \"ornaments\": [\"Hieroglyphic carvings\", \"Colossal anthropomorphic sculptures\"],\n  \"color_palette\": [\"Ochre\", \"Sand\", \"Terracotta\", \"Azure\"],\n  \"location_hint\": \"Egypt / Nile Valley\",\n  \"summary\": \"A composite representation of Ancient Egyptian monumental architecture featuring pyramids, the Sphinx, and temple complexes set against a desert cliff backdrop.\"\n}\n```"
}


,id,image_path,hybrid_score,image_score,metadata_score
0,0,D:\dev\visquery\ml\test-data\AncientEgyptianAr...,0.996642,1.000831,0.990360
1,1,D:\dev\visquery\ml\test-data\AncientGreekArchi...,0.592257,0.555252,0.647766
2,8,D:\dev\visquery\ml\test-data\IslamicArchitectu...,0.573683,0.510036,0.669155
3,9,D:\dev\visquery\ml\test-data\MannerismArchitec...,0.540223,0.438786,0.692377
4,14,D:\dev\visquery\ml\test-data\PersianIranianArc...,0.518710,0.479759,0.577137


In [21]:
def rag_with_metadata(query_text: str, retrieved_df: pd.DataFrame) -> str:
    contexts = []
    for _, row in retrieved_df.iterrows():
        ctx = {
            "id": int(row["id"]),
            "image_path": row["image_path"],
            "metadata": row["metadata"],
        }
        contexts.append(ctx)

    system_prompt = (
        "You are an architecture assistant. Answer strictly from retrieved metadata context. "
        "If uncertain, state the limitation clearly."
    )
    user_prompt = (
        f"Question: {query_text}\n\n"
        f"Retrieved Context JSON: {json.dumps(contexts, ensure_ascii=False)}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    return ollama_chat(RAG_LLM_MODEL, messages, temperature=0.2)


In [22]:
# RAG demo
question = "Identify likely architectural style and key visual elements from similar matches."
rag_answer = rag_with_metadata(question, result_df)
print(rag_answer)


Based on the provided metadata, here are the architectural styles and their key visual elements:

*   **Ancient Egyptian**:
    *   **Structures**: Pyramids, Hypostyle halls, Colossal statues, and Courtyards.
    *   **Ornaments**: Hieroglyphic carvings, Sphinx, and Columns.
*   **Classical Greek**:
    *   **Structures**: Doric and Ionic columns, pediments, peristyle, and amphitheaters.
    *   **Ornaments**: Fluting, entablatures, and friezes.
*   **Islamic / Neo-Islamic**:
    *   **Structures**: Central dome with flanking smaller domes, pointed arches, and colonnades.
    *   **Ornaments**: Geometric arabesque patterns, Crescents, and intricate tilework.
*   **Timurid / Islamic**:
    *   **Structures**: Symmetrical iwan, ribbed dome, twin minarets, and courtyards.
    *   **Ornaments**: Geometric mosaics, arabesque patterns, and calligraphy.
*   **Renaissance / Mannerism**:
    *   **Structures**: Symmetrical facade with central arched gateway and flanking niches.
    *   **Orname

## Notes for Production

- Use VLM (`OLLAMA_VLM_MODEL`) offline/batch to pre-generate metadata for performance and cost control.
- Use a faster LLM (`OLLAMA_RAG_LLM_MODEL`) online for low-latency RAG answers.
- Persist `metadata` as JSONB and vectors in your production DB (e.g., Postgres + pgvector or FAISS service).
- Reuse this two-stage retrieval: image embedding rank -> metadata embedding rerank.
